In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

# --- Load Data and Create a Unified DataFrame ---
PKL_PATH = '../../data/supplementary_figures/slr_res/DIVA_data/Coast_with_Mang.pkl' 
COASTLINE_DATA_PATH = "../../data/supplementary_figures/slr_res/global_coastal_wetland_model/Results/coastline_shp/DIVA_coastline_WGS84_Point.csv"
with open(PKL_PATH, 'rb') as f:
    data = pickle.load(f)

# # Extract relevant data
df = pd.DataFrame({
    'ClsFID': data['ClsFID'],
    'area': data['area'],
    'agb_present': data['agb_present'],
    'agb_grow': data['agb_grow'],
    'agb_126': data['agb_126'],
    'agb_585': data['agb_585'],
    # 注意：根据您的脚本，这里选择的是第2列(Pop5)和第1列(50%恢复)
    'area_after_SLR_126': data['area_after_SLR_126'][:, 1], 
    'area_after_SLR_585': data['area_after_SLR_585'][:, 1],
    'area_restoration_126': data['area_restoration_126'][:, 0],
    'area_restoration_585': data['area_restoration_585'][:, 0],
})

# --- Load and Merge Country Information ---
# 从新的基础数据文件中读取国家信息
coastline_data = pd.read_csv(COASTLINE_DATA_PATH)
# 使用.merge()方法，通过'ClsFID'将国家信息精确地匹配到每一行
# Parse the 'ID_Country' column to extract the clean country name.
# Split the column on the first space. expand=True creates new columns.
country_split = coastline_data["ID_Country"].str.split(" ", n=1, expand=True)
# Create the country_info DataFrame for merging.
# We take the original CLSFID for merging and the newly extracted country name (column 1).
country_info = pd.DataFrame({'ClsFID': coastline_data['CLSFID'], 'Country': country_split[1]})
# 将主数据框与国家信息合并
df = pd.merge(df, country_info, on='ClsFID', how='left')

# --- Calculate AGB Changes (Vectorized Operations) ---
# 定义一个函数，使其在DataFrame上操作，更清晰
def calculate_agb_changes(df, scenario_suffix):
    agb_scen = f'agb_{scenario_suffix}'
    area_slr_scen = f'area_after_SLR_{scenario_suffix}'
    area_res_scen = f'area_restoration_{scenario_suffix}'
    # 直接在DataFrame上创建新列，单位转换为 Tg (10^6 Mg)
    df[f'total_change_{scenario_suffix}'] = (df[agb_scen] * (df[area_slr_scen] + df[area_res_scen]) - df['agb_present'] * df['area']) / 1e6
    df[f'grow_change_{scenario_suffix}'] = (df['agb_grow'] * df['area'] - df['agb_present'] * df['area']) / 1e6
    df[f'ht_change_{scenario_suffix}'] = (df[agb_scen] * df['area'] - df['agb_grow'] * df['area']) / 1e6
    df[f'res_change_{scenario_suffix}'] = (df[agb_scen] * df[area_res_scen]) / 1e6
    df[f'slr_change_{scenario_suffix}'] = (df[agb_scen] * (df[area_slr_scen] - df['area'])) / 1e6
    return df
df = calculate_agb_changes(df, '126')
df = calculate_agb_changes(df, '585')

# --- Aggregate Results by Country using groupby ---
agg_cols_126 = {
    'AGB_Change': 'total_change_126',
    'AGB_Change_Grow': 'grow_change_126',
    'AGB_Change_HT': 'ht_change_126',
    'AGB_Change_RES': 'res_change_126',
    'AGB_Change_SLR': 'slr_change_126'
}
agg_cols_585 = {
    'AGB_Change': 'total_change_585',
    'AGB_Change_Grow': 'grow_change_585',
    'AGB_Change_HT': 'ht_change_585',
    'AGB_Change_RES': 'res_change_585',
    'AGB_Change_SLR': 'slr_change_585'
}
# Invert the dictionaries for the rename function.
rename_map_126 = {v: k for k, v in agg_cols_126.items()}
rename_map_585 = {v: k for k, v in agg_cols_585.items()}

# 使用groupby().sum()一步完成按国家汇总，然后使用翻转后的字典进行重命名
country_agb_change_126 = df.groupby('Country', as_index=False)[list(agg_cols_126.values())].sum().rename(columns=rename_map_126).sort_values(by='AGB_Change')
country_agb_change_585 = df.groupby('Country', as_index=False)[list(agg_cols_585.values())].sum().rename(columns=rename_map_585).sort_values(by='AGB_Change')



In [ ]:
# The unit is already Mg, we sum it up and convert to Tg
df['agb_present_total'] = df['agb_present'] * df['area'] / 1e6
country_present_agb = df.groupby('Country', as_index=False)['agb_present_total'].sum()

# Merge present-day AGB with the change data
country_analysis_126 = pd.merge(country_agb_change_126, country_present_agb, on='Country')
country_analysis_585 = pd.merge(country_agb_change_585, country_present_agb, on='Country')

import matplotlib.pyplot as plt
from brokenaxes import brokenaxes
from adjustText import adjust_text

# --- Set plotting style ---
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 16

# --- Create the figure ---
fig = plt.figure(figsize=(12, 6))
fig.supxlabel('Present-day Mangrove AGB (Tg DM)', y=0, fontsize=16)

# --- Define constants ---
break_point = 190
xlims = ((0, break_point), (490, 540))
top_10_countries = country_analysis_126.sort_values(by='agb_present_total', ascending=False).head(10)['Country'].tolist()

# --- Create the broken axes objects ---
gs = fig.add_gridspec(1, 2, wspace=0.1)
bax1 = brokenaxes(xlims=xlims, subplot_spec=gs[0, 0])
bax2 = brokenaxes(xlims=xlims, subplot_spec=gs[0, 1])

# --- Plot data on the broken axes ---
bax1.scatter(country_analysis_126['agb_present_total'], country_analysis_126['AGB_Change'], alpha=0.8, color='royalblue', edgecolors='k', s=60, zorder=10)
bax1.axhline(0, color='grey', linestyle='--', linewidth=1.2)
bax1.set_ylabel('AGB Change (Tg DM)', labelpad=50) # Reduced labelpad for better look
bax1.set_title('a', loc='left', fontsize=16, fontweight='bold')
bax1.grid(linestyle=':', alpha=0.7)
bax1.text(0.05, 0.95, 'Low-warming scenario', transform=bax1.axs[0].transAxes, ha='left', fontsize=16)

bax2.scatter(country_analysis_585['agb_present_total'], country_analysis_585['AGB_Change'], alpha=0.8, color='firebrick', edgecolors='k', s=60, zorder=10)
bax2.axhline(0, color='grey', linestyle='--', linewidth=1.2)
bax2.set_title('b', loc='left', fontsize=16, fontweight='bold')
bax2.grid(linestyle=':', alpha=0.7)
bax2.text(0.05, 0.95, 'High-warming scenario', transform=bax2.axs[0].transAxes, ha='left', fontsize=16)
bax2.tick_params(axis='y', labelleft=False)

# --- Synchronize the Y-axis limits ---
y1_min, y1_max = bax1.axs[0].get_ylim()
y2_min, y2_max = bax2.axs[0].get_ylim()
bax1.set_ylim(bottom=min(y1_min, y2_min), top=max(y1_max, y2_max)+20)
bax2.set_ylim(bottom=min(y1_min, y2_min), top=max(y1_max, y2_max)+20)

# --- REFACTORED HYBRID LABELING STRATEGY ---

# *** VVVV 您唯一需要交互的地方 VVVV ***
manual_overrides = {
    'low_warming': {
        'Myanmar': {'xytext': (15, 15)},
        'Bangladesh': {'xytext': (-20, 10)},
        'Colombia': {'xytext': (0, -25)},
        'Mexico': {'xytext': (0, -25)},
        'Venezuela': {'xytext': (15, -25)},
        'Brazil': {'xytext': (0, -25)},
        'Papua New Guinea': {'xytext': (45, 7)},
    },
    'high_warming': {
        'Bangladesh': {'xytext': (15, 35)},
        'Colombia': {'xytext': (15, -35)},
        'Venezuela': {'xytext': (15, -25)},
        'Malaysia': {'xytext': (15, 5)},
        'Papua New Guinea': {'xytext': (75, 5)},
    }
}
# *** ^^^^ 您只需要在这里修改 ^^^^ ***

def process_labels(df, bax, overrides_key, break_point):
    """
    A reusable function to handle both manual and automatic labeling for a given plot.
    """
    texts_to_adjust = []
    points_to_avoid = df[df['agb_present_total'] < break_point]
    overrides = manual_overrides.get(overrides_key, {})

    for _, row in df.iterrows():
        if row['Country'] in top_10_countries:
            source_ax = bax.axs[1] if row['agb_present_total'] > break_point else bax.axs[0]
            if row['Country'] in overrides:
                # Manual placement
                override = overrides[row['Country']]
                source_ax.annotate(row['Country'], xy=(row['agb_present_total'], row['AGB_Change']),
                                   xytext=override.get('xytext', (5, 5)), textcoords='offset points',
                                   ha='center', va='bottom', fontsize=12,
                                   arrowprops=dict(arrowstyle="-", color='black', lw=0.5))
            elif row['agb_present_total'] < break_point:
                # Add to auto-adjust list (crowded area)
                texts_to_adjust.append(source_ax.text(row['agb_present_total'], row['AGB_Change'], row['Country'], fontsize=12))
            else:
                # Direct placement (uncrowded area)
                source_ax.text(row['agb_present_total'], row['AGB_Change'] + 2, row['Country'], ha='center', va='bottom', fontsize=12)
    
    return texts_to_adjust, points_to_avoid

# --- Process both plots using the new function ---
texts_126, points_126 = process_labels(country_analysis_126, bax1, 'low_warming', break_point)
texts_585, points_585 = process_labels(country_analysis_585, bax2, 'high_warming', break_point)

# --- Run adjust_text for each plot ---
adjust_text(texts_126, x=points_126['agb_present_total'], y=points_126['AGB_Change'],
            ax=bax1.axs[0], expand_points=(2.5, 2.5),
            arrowprops=dict(arrowstyle="-", color='black', lw=0.5))

adjust_text(texts_585, x=points_585['agb_present_total'], y=points_585['AGB_Change'],
            ax=bax2.axs[0], expand_points=(2.5, 2.5),
            arrowprops=dict(arrowstyle="-", color='black', lw=0.5))

plt.savefig('../../figures/supplementary/figS11_present_agb_vs_future_change.png', dpi=300, bbox_inches='tight')
plt.show()